# Assignment 2: From Trees to Neural Networks

**Dataset:** UCI Bank Marketing (`bank-additional-full.csv`)

**Objective:** Compare gradient-boosted decision trees (XGBoost) and a multi-layer perceptron (MLP) on a tabular binary classification task: predict whether a client subscribes to a term deposit (`y = "yes"` is the positive class).

This section covers Step 1 — data preparation: audit, cleaning decisions, a stratified 70/15/15 split, and preprocessing fitted on training data only.

**How to run in Google Colab**
1. Upload this notebook to Colab.
2. Upload `bank-additional-full.csv` into the Colab working directory (left sidebar → Files → Upload).
3. Runtime → Run all.

## 0. Setup

In [ ]:
import os
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_FILE = "bank-additional-full.csv"
TARGET = "y"
POSITIVE_LABEL = "yes"
LEAKAGE_COLUMNS = ["duration"]
PDAYS_NO_CONTACT = 999
PDAYS_NO_CONTACT_ENCODED = -1

print(f"Python:       {platform.python_version()}")
print(f"pandas:       {pd.__version__}")
print(f"numpy:        {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"Platform:     {platform.platform()}")
print(f"CPU cores:    {os.cpu_count()}")
print(f"RANDOM_STATE: {RANDOM_STATE}")

### 0.1 Load the data

Place `bank-additional-full.csv` in the same working directory as this notebook. Source: UCI Machine Learning Repository, Bank Marketing dataset — `bank-additional-full.csv` from `bank-additional.zip`. The file is semicolon-separated.

In [ ]:
DATA_PATH = Path(DATA_FILE)
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_FILE} not found in the working directory. "
        "In Colab: Files → Upload → select bank-additional-full.csv, then re-run this cell."
    )

print(f"Found: {DATA_PATH.resolve()}")

## 1. Data audit

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")
print(f"Rows: {df.shape[0]:,}  Columns: {df.shape[1]}")
df.head()

### 1.1 Feature types

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = [c for c in df.columns if c not in numeric_cols and c != TARGET]

feature_types = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
}).drop(index=TARGET)
feature_types["type"] = np.where(feature_types.index.isin(numeric_cols), "numeric", "nominal categorical")

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
feature_types

### 1.2 Target distribution

In [ ]:
target_counts = df[TARGET].value_counts()
target_share = df[TARGET].value_counts(normalize=True)
pd.DataFrame({"count": target_counts, "proportion": target_share.round(4)})

### 1.3 Missing and special values

In [ ]:
# pandas nulls vs. the dataset's literal "unknown" category
null_counts = df.isna().sum()
print(f"Total pandas-null cells: {int(null_counts.sum())}")

unknown_summary = pd.DataFrame({
    "unknown_count": [(df[c] == "unknown").sum() for c in categorical_cols],
}, index=categorical_cols)
unknown_summary["unknown_pct"] = (100 * unknown_summary["unknown_count"] / len(df)).round(2)
unknown_summary.sort_values("unknown_count", ascending=False)

In [ ]:
# Positive rate when a field is "unknown" vs. known: is "unknown" informative?
rows = []
for col in unknown_summary.index[unknown_summary["unknown_count"] > 0]:
    is_unknown = df[col] == "unknown"
    rows.append({
        "column": col,
        "positive_rate_unknown": (df.loc[is_unknown, TARGET] == POSITIVE_LABEL).mean(),
        "positive_rate_known": (df.loc[~is_unknown, TARGET] == POSITIVE_LABEL).mean(),
    })
pd.DataFrame(rows).set_index("column").round(4)

In [ ]:
no_contact = df["pdays"] == PDAYS_NO_CONTACT
print(f"pdays == {PDAYS_NO_CONTACT}: {no_contact.sum():,} rows ({no_contact.mean():.2%})")
print("pdays range when a previous contact exists:",
      df.loc[~no_contact, "pdays"].min(), "to", df.loc[~no_contact, "pdays"].max())
print()
print(pd.crosstab(no_contact.rename("pdays == 999"), df["poutcome"]))

In [ ]:
print(f"Exact duplicate rows (all columns):           {df.duplicated().sum():,}")
print(f"Duplicate rows after excluding duration:      {df.drop(columns=LEAKAGE_COLUMNS).duplicated().sum():,}")
print(f"Rows where default == 'yes':                  {(df['default'] == 'yes').sum():,}")

**Audit findings.**

- The file has 41,188 rows and 21 columns: 10 numeric predictors, 10 nominal categorical predictors, and the target `y`.
- The positive class is rare: about 11.3% of clients subscribed. Accuracy alone would be misleading (predicting "no" for everyone scores about 88.7%), which is why Average Precision is the primary metric.
- There are no pandas-null values. Missingness appears instead as the literal category `"unknown"` in six columns, most heavily in `default` (about 20.9% of rows).
- `"unknown"` is informative rather than random noise: for example, clients with `default = "unknown"` subscribe at a much lower rate than clients with a known value.
- `pdays = 999` is a sentinel meaning *no previous contact* and covers about 96.3% of rows. Real values range from 0 to 27 days. It aligns with `poutcome`: every `success` row has a real `pdays`, while every `nonexistent` row has `pdays = 999`.
- There are 12 exact duplicate rows. Without an identifier there is no way to tell whether they are repeated records or different clients with identical attributes.

## 2. Cleaning decisions and target definition

| Item | Decision | Reason |
|---|---|---|
| Target | `y = "yes"` → 1, `"no"` → 0 | Positive class defined by the handout |
| `duration` | Dropped from predictors | Call length is only known after the call ends; using it would leak information unavailable at prediction time |
| `pdays = 999` | New indicator `previously_contacted`; `pdays` set to −1 when there was no previous contact | 999 is not an elapsed number of days. Leaving it would put no-contact clients far from real values (0–27) and dominate the scale of the column. The indicator states the meaning explicitly, and −1 sits just outside the valid range so both models can separate the two cases |
| Literal `"unknown"` | Kept as its own category and one-hot encoded | It is a recorded value, not a blank cell, and its subscription rate differs from known values. Imputing the mode would erase that signal and invent information |
| Missing values (NaN) | No imputation needed | The audit found no pandas-null values |
| Duplicates | All rows kept | Only 12 exact duplicates (0.03%) and no client ID to confirm they are repeats; keeping all rows preserves the dataset described in the handout |
| `default = "yes"` | Kept | Only 3 rows; the encoder ignores the level if it never appears in training |

These steps are deterministic, row-by-row rules that learn nothing from the data, so applying them before the split does not leak information.

In [ ]:
def prepare_features(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Deterministic, row-wise cleaning only (no statistics learned from the data)."""
    data = raw.copy()
    y = (data[TARGET] == POSITIVE_LABEL).astype(int).rename("target")

    data["previously_contacted"] = (data["pdays"] != PDAYS_NO_CONTACT).astype(int)
    data["pdays"] = data["pdays"].where(data["previously_contacted"] == 1, PDAYS_NO_CONTACT_ENCODED)

    X = data.drop(columns=[TARGET, *LEAKAGE_COLUMNS])
    return X, y


X, y = prepare_features(df)

assert "duration" not in X.columns
assert TARGET not in X.columns
assert (X.loc[X["previously_contacted"] == 0, "pdays"] == PDAYS_NO_CONTACT_ENCODED).all()

print(f"X: {X.shape}  y: {y.shape}  positive rate: {y.mean():.4f}")
X[["pdays", "previously_contacted", "previous", "poutcome"]].head()

In [ ]:
CATEGORICAL_FEATURES = [
    "job", "marital", "education", "default", "housing", "loan",
    "contact", "month", "day_of_week", "poutcome",
]
NUMERIC_FEATURES = [
    "age", "campaign", "pdays", "previous", "previously_contacted",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed",
]

assert set(CATEGORICAL_FEATURES) | set(NUMERIC_FEATURES) == set(X.columns)
assert not set(CATEGORICAL_FEATURES) & set(NUMERIC_FEATURES)
print(f"{len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical = {X.shape[1]} predictors")

## 3. Stratified 70/15/15 train / validation / test split

The split is made in two stages with `random_state = 42`, stratified on the target each time so all three partitions keep the same positive-class rate.

- **Train (70%)** — fit preprocessing and models.
- **Validation (15%)** — XGBoost early stopping and choosing configurations by validation Average Precision.
- **Test (15%)** — used once, after both final configurations are fixed.

The same split rows are reused for every GBDT and MLP experiment.

In [ ]:
TRAIN_SIZE, VALID_SIZE, TEST_SIZE = 0.70, 0.15, 0.15

# Stage 1: 70% train vs. 30% holdout. Stage 2: split the holdout 50/50 into validation and test.
X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, train_size=TRAIN_SIZE, stratify=y, random_state=RANDOM_STATE
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_hold, y_hold, test_size=TEST_SIZE / (VALID_SIZE + TEST_SIZE), stratify=y_hold, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_valid), len(y_test)],
    "share": [len(y_train) / len(y), len(y_valid) / len(y), len(y_test) / len(y)],
    "positives": [y_train.sum(), y_valid.sum(), y_test.sum()],
    "positive_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=["train", "validation", "test"]).round(4)
split_summary

In [ ]:
train_idx, valid_idx, test_idx = set(X_train.index), set(X_valid.index), set(X_test.index)

assert not train_idx & valid_idx and not train_idx & test_idx and not valid_idx & test_idx
assert len(train_idx | valid_idx | test_idx) == len(X)
assert X_train.index.equals(y_train.index) and X_valid.index.equals(y_valid.index) and X_test.index.equals(y_test.index)
print("Splits are disjoint, cover every row, and X/y indices are aligned.")

**Interpretation.** The positive-class rate is essentially identical across train, validation, and test, so differences between them later reflect the models rather than an uneven split. A random split measures performance on clients drawn from the same historical campaigns; it does not test generalization to future campaigns. The dataset is also a historical benchmark (2008–2010 Portuguese bank campaigns), so scores here are not evidence of current banking performance.

## 4. Leakage-safe preprocessing

Learned preprocessing is fitted on `X_train` only and then applied unchanged to validation and test:

- **One-hot encoding** for all 10 nominal categorical columns, with `handle_unknown="ignore"` so a category that never appears in training becomes an all-zero row instead of an error.
- **Standard scaling** of numeric columns for the MLP only. Gradient-based networks train poorly when inputs have very different scales (for example `nr.employed` around 5,000 versus `previously_contacted` in {0, 1}). Tree splits depend only on the ordering of values, so the GBDT input keeps the raw numeric values.

Both inputs contain exactly the same columns and rows; the only model-specific difference is numeric scaling. Preprocessing time is recorded separately from model fitting time.

In [ ]:
def make_preprocessor(scale_numeric: bool) -> ColumnTransformer:
    numeric_step = StandardScaler() if scale_numeric else "passthrough"
    return ColumnTransformer(
        transformers=[
            ("num", numeric_step, NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")


def fit_transform_splits(preprocessor: ColumnTransformer):
    """Fit on training rows only; reuse the fitted transformer for validation and test."""
    start = time.perf_counter()
    train_t = preprocessor.fit_transform(X_train)
    valid_t = preprocessor.transform(X_valid)
    test_t = preprocessor.transform(X_test)
    return train_t, valid_t, test_t, time.perf_counter() - start


tree_preprocessor = make_preprocessor(scale_numeric=False)
mlp_preprocessor = make_preprocessor(scale_numeric=True)

X_train_tree, X_valid_tree, X_test_tree, tree_preprocess_sec = fit_transform_splits(tree_preprocessor)
X_train_mlp, X_valid_mlp, X_test_mlp, mlp_preprocess_sec = fit_transform_splits(mlp_preprocessor)

PREPROCESSING_TIME_SEC = {"GBDT (one-hot only)": tree_preprocess_sec, "MLP (one-hot + scaling)": mlp_preprocess_sec}

pd.DataFrame({
    "train": [X_train_tree.shape, X_train_mlp.shape],
    "validation": [X_valid_tree.shape, X_valid_mlp.shape],
    "test": [X_test_tree.shape, X_test_mlp.shape],
    "preprocessing_sec": [round(tree_preprocess_sec, 4), round(mlp_preprocess_sec, 4)],
}, index=["GBDT input", "MLP input"])

In [ ]:
# 1. Both models see the same encoded columns (same underlying information).
assert list(X_train_tree.columns) == list(X_train_mlp.columns)

# 2. No missing values after preprocessing.
for frame in [X_train_tree, X_valid_tree, X_test_tree, X_train_mlp, X_valid_mlp, X_test_mlp]:
    assert not frame.isna().any().any()

# 3. Tree numeric columns are unchanged; MLP numeric columns are standardized with training statistics.
assert np.allclose(X_train_tree[NUMERIC_FEATURES].to_numpy(), X_train[NUMERIC_FEATURES].to_numpy())

scaling_check = pd.DataFrame({
    "train_mean": X_train_mlp[NUMERIC_FEATURES].mean(),
    "train_std": X_train_mlp[NUMERIC_FEATURES].std(ddof=0),
    "valid_mean": X_valid_mlp[NUMERIC_FEATURES].mean(),
    "test_mean": X_test_mlp[NUMERIC_FEATURES].mean(),
}).round(4)

n_onehot = len(X_train_tree.columns) - len(NUMERIC_FEATURES)
print(f"Encoded features: {X_train_tree.shape[1]} ({len(NUMERIC_FEATURES)} numeric + {n_onehot} one-hot)")
print("Categories with no training rows (ignored if seen later):",
      [f"{col}={val}" for col in CATEGORICAL_FEATURES for val in X[col].unique()
       if val not in set(X_train[col])] or "none")
scaling_check

**Interpretation.** Both models receive the same encoded features, and nothing is missing. The scaled MLP features have mean 0 and standard deviation 1 on the training rows only; validation and test means are close to, but not exactly, 0. That small difference is expected: it shows the scaler reused training statistics instead of being refit on held-out data.

**Step 1 outputs used later:** `X_train_tree / X_valid_tree / X_test_tree` (GBDT), `X_train_mlp / X_valid_mlp / X_test_mlp` (MLP), `y_train / y_valid / y_test`, and `PREPROCESSING_TIME_SEC`.